# 💬 11장 · 리뷰 속 마음 읽기: 텍스트 분석과 언어 AI (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 11장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호와 제목은 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 11-1, 11-3, 11-4, 11-5, 11-7, 11-8 |
| ☆ 선택 | 자습이나 심화 | 11-2, 11-6, 11-9, 11-10, 11-11 |

**데이터** (모두 GitHub에서 바로 읽습니다)
- 네이버 영화 리뷰 말뭉치(NSMC): 훈련 15만 개, 시험 5만 개. 저장소 `e9t/nsmc`
- 연방주의자 논문 85편: R corpus 패키지의 `federalist.rda`. 저장소 `cran/corpus`
- 미니 프로젝트: 네이버 쇼핑 리뷰 20만 개. 저장소 `bab2min/corpus`

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. 리뷰 20만 개를 형태소로 자르므로 전체 실행에 10분 남짓 걸립니다.

## 0 · 준비

형태소 분석기 **키위(kiwipiepy)**, 단어 임베딩 도구 **젠심(gensim)**, R 데이터 파일을 읽는 **파이리더(pyreadr)**, 그래프의 한글을 살리는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q kiwipiepy gensim pyreadr koreanize-matplotlib

## 05. 파이썬 실습: 영화 리뷰의 감정 맞히기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 데이터는 인터넷 주소에서 바로 읽습니다. 코랩에는 키위, 단어 임베딩 도구 젠심(gensim), R 데이터 파일을 읽는 파이리더(pyreadr)가 없습니다. 먼저 `!pip install kiwipiepy gensim pyreadr koreanize-matplotlib`로 설치합니다. 리뷰 20만 개를 형태소로 자르므로 전체 실습에 10분 남짓 걸립니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 11-1 ★] 리뷰 불러오기와 정리

영화 리뷰를 읽어 빈 리뷰, 겹친 리뷰, 라벨이 엇갈린 리뷰를 정리하고 기준선을 구하시오. "굿", "good" 같은 짧은 리뷰에 붙은 라벨도 세어 보시오.

① 한글 글꼴을 설정하고 데이터를 읽습니다. ② 빈 리뷰와 라벨이 엇갈린 글을 지우고 겹친 글은 하나만 남깁니다. ③ 짧은 리뷰 네 개의 라벨을 셉니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

BASE = "https://raw.githubusercontent.com/e9t/nsmc/master/"
train0 = pd.read_csv(BASE + "ratings_train.txt", sep="\t", quoting=3)
test0 = pd.read_csv(BASE + "ratings_test.txt", sep="\t", quoting=3)
print("훈련", train0.shape, " 시험", test0.shape)
print(train0.head(3).to_string())
print("빈 리뷰:", train0["document"].isna().sum(), "/", test0["document"].isna().sum())

train = train0.dropna(subset=["document"]).copy()           # 빈 리뷰 지우기
test = test0.dropna(subset=["document"]).copy()
n_label = train.groupby("document")["label"].nunique()      # 같은 글에 붙은 라벨 종류 수
conflict = n_label[n_label > 1].index                       # 2종류 = 긍정과 부정이 모두 붙은 글
print("같은 글이 다시 나온 횟수:", train["document"].duplicated().sum())
print("같은 글인데 라벨이 엇갈린 글:", len(conflict))
train = train[~train["document"].isin(conflict)].drop_duplicates("document")
print("정리 후 훈련:", len(train), " 시험:", len(test))
y_tr, y_te = train["label"].values, test["label"].values
print("시험 리뷰의 긍정 비율:", round(y_te.mean(), 3))
print("기준선(많은 쪽으로 찍기):", round(max(y_te.mean(), 1 - y_te.mean()), 3))

top = train0[train0["document"].isin(["굿", "good", "최고", "재밌다"])]
cnt = pd.crosstab(top["document"], top["label"])             # 글마다 부정(0), 긍정(1) 개수
print(cnt.to_string())
cnt.plot.barh(stacked=True, color=["#C8234B", "#BBBBBB"], figsize=(6, 3))
plt.legend(["부정 라벨", "긍정 라벨"])
plt.xlabel("훈련 파일에 나온 횟수")
plt.ylabel("")
plt.show()

훈련 리뷰 146,025개, 시험 리뷰 49,997개가 남았습니다. 시험 리뷰의 긍정 비율은 0.503이라서 찍기만 해도 50.3%를 맞힙니다. "good"은 긍정 42번, 부정 50번으로 부정 라벨이 더 많았습니다. 라벨이 별점에서 왔다는 것을 다시 보여 줍니다.

> **[코드 읽기] `groupby( )["label"].nunique( )`와 `isin( )`**
>
> `groupby("document")`는 같은 글끼리 묶고, `["label"].nunique()`는 묶음마다 라벨이 몇 종류인지 셉니다. 2가 나온 글은 긍정과 부정이 모두 붙은 글입니다. `isin(목록)`은 값이 목록 안에 있으면 True를 냅니다. 앞에 `~`를 붙이면 "목록에 없는 것"이 됩니다.

#### [파이썬 실습 11-2 ☆] 리뷰 길이, 기호, 다음에 오는 말

리뷰 길이를 요약하고, 기호와 단어가 든 리뷰의 긍정 비율을 구하시오. "시간" 다음에 오는 단어도 세어 보시오.

In [ ]:
L = train["document"].str.len()
print("길이 평균", round(L.mean(), 1), " 가운데", int(L.median()), " 최대", L.max(),
      " 10자 이하", round((L <= 10).mean(), 3))
print("라벨별 평균 길이:", L.groupby(train["label"]).mean().round(1).to_dict())
rate = {}
for pat in ["최고", "!", "ㅠㅠ|ㅜㅜ", "ㅋㅋ", r"\.\.\.", r"\?", "ㅡㅡ", "쓰레기"]:
    has = train["document"].str.contains(pat)
    key = pat.replace("\\", "")                              # 그래프 이름표에서 역슬래시 빼기
    rate[key] = train.loc[has, "label"].mean()
    print(f"{key:6s} {has.sum():6d}개  긍정 비율 {rate[key]:.3f}")

pd.Series(rate).sort_values().plot.barh(color="#BBBBBB", figsize=(6, 3.5))
plt.axvline(0.5, color="#C8234B", ls="--")
plt.xlabel("이것이 든 리뷰의 긍정 비율")
plt.show()

nxt = pd.Series([t[i + 1] for t in train["document"].str.split()      # "시간" 바로 다음에 온 단어
                 for i in range(len(t) - 1) if t[i] == "시간"])
print('"시간" 다음 단어', len(nxt), "개:", nxt.value_counts().head(5).to_dict())

"ㅠㅠ"가 든 리뷰의 70.8%가 긍정이고, "ㅡㅡ"가 든 리뷰는 10.6%만 긍정입니다. `str.contains( )`의 패턴은 정규식이라서 "ㅠㅠ|ㅜㅜ"는 "둘 가운데 하나"를 뜻합니다. 마침표 세 개와 물음표는 정규식에서 특별한 뜻이 있으므로 앞에 `\`를 붙였습니다.

#### [파이썬 실습 11-3 ★] 형태소로 자르기

키위로 리뷰를 형태소로 자르고, 띄어쓰기로 자를 때와 조각의 종류 수를 견주시오.

① 예문 하나를 띄어쓰기와 키위로 잘라 봅니다. ② 모든 리뷰를 형태소로 잘라 "m" 열에 둡니다. ③ 두 방법의 조각 종류 수와 "영화"로 시작하는 단어 꼴을 셉니다.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
ex = "이 영화 진짜 재미없지는 않았어요ㅋㅋ 배우들 연기가 최고!"
print(ex.split())
print([(t.form, t.tag) for t in kiwi.tokenize(ex)])
print([t.form for t in kiwi.tokenize("너무재밓었다그래서보는것을추천한다")])

def morphs(texts):                                           # 문장 목록 -> "형태소 형태소 ..." 목록
    return [" ".join(t.form for t in toks) for toks in kiwi.tokenize(texts)]

train["m"] = morphs(train["document"].tolist())              # 1~2분 걸립니다
test["m"] = morphs(test["document"].tolist())
print(train[["document", "m"]].head(2).to_string())

ws = train["document"].str.split().explode()                 # 띄어쓰기 단어
ms = train["m"].str.split().explode()                        # 형태소
print("띄어쓰기 단어 종류:", ws.nunique(), " 형태소 종류:", ms.nunique())
print("한 번만 나온 종류의 비율:", round((ws.value_counts() == 1).mean(), 3), round((ms.value_counts() == 1).mean(), 3))
yh = ws[ws.str.fullmatch(r"영화[가-힣]{0,3}")].value_counts()
print("'영화'로 시작하는 단어 꼴:", len(yh), yh.head(5).to_dict())

kinds = pd.Series({"띄어쓰기 단어": ws.nunique(), "형태소": ms.nunique()})
kinds.plot.bar(color=["#BBBBBB", "#C8234B"], figsize=(4, 3))
plt.ylabel("종류 수")
plt.xticks(rotation=0)
plt.show()

키위는 "재미없지는"을 "재미없(VA, 형용사) / 지 / 는"으로, "않았어요ㅋㅋ"를 "않 / 었 / 어요 / ㅋㅋ"로 나눴습니다. 띄어쓰기 단어는 79.6%가 한 번만 나왔고, 형태소는 52.0%였습니다. 리뷰 20만 개를 자르는 데 1~2분이 걸립니다.

> **[코드 읽기] `kiwi.tokenize( )`와 `str.split( ).explode( )`**
>
> `kiwi.tokenize(문장)`은 형태소 조각의 목록을 돌려줍니다. 조각마다 `.form`(글자)과 `.tag`(품사)가 있습니다. 문장 목록을 한꺼번에 넣으면 더 빠릅니다. `str.split()`은 리뷰를 띄어쓰기 단어의 목록으로 바꾸고, `explode()`는 그 목록을 한 줄에 하나씩 풀어 놓습니다. 그래서 `nunique()`로 종류를 셀 수 있습니다.

#### [파이썬 실습 11-4 ★] TF-IDF와 코사인 유사도 확인하기

[예제 11-1]의 리뷰 세 개로 문서-단어 행렬, TF-IDF, 코사인 유사도를 구하고 손 계산과 견주시오.

① `CountVectorizer`로 개수를 셉니다. ② (식 11-1)대로 idf를 곱합니다. ③ `cosine_similarity`로 세 리뷰 사이의 유사도를 구합니다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["배우 연기 최고", "배우 연기 지루", "스토리 지루 지루"]
cv = CountVectorizer(token_pattern=r"\S+")
X = cv.fit_transform(docs).toarray()                         # 문서-단어 행렬(개수)
print(cv.get_feature_names_out())
print(X)
idf = np.log(len(docs) / (X > 0).sum(axis=0))                # idf = ln(N / df)
print("idf:", idf.round(3))
T = X * idf                                                  # TF-IDF = 개수 × idf
print(T.round(3))
print("d1·d2 =", round(T[0] @ T[1], 3), " 길이:", np.linalg.norm(T, axis=1).round(3))
S = cosine_similarity(T)
print("코사인 유사도:\n", S.round(3))

plt.figure(figsize=(4, 3.4))
plt.imshow(S, cmap="Greys", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{S[i, j]:.3f}", ha="center", va="center", color="#C8234B")
plt.xticks(range(3), ["d1", "d2", "d3"])
plt.yticks(range(3), ["d1", "d2", "d3"])
plt.show()

idf, TF-IDF 벡터, 코사인 유사도가 [예제 11-1]의 손 계산과 같습니다. d1과 d3은 겹치는 단어가 없어 0입니다. `token_pattern=r"\S+"`는 "공백이 아닌 글자 덩어리를 한 토큰으로"라는 뜻입니다. 이것을 빼면 사이킷런이 한 글자짜리 토큰을 버립니다.

#### [파이썬 실습 11-5 ★] TF-IDF와 로지스틱 회귀로 감정 분류하기

띄어쓰기 단어와 형태소 1~2개씩으로 각각 TF-IDF를 만들어 로지스틱 회귀를 맞추고, 정확도, 혼동행렬, 정밀도, 재현율을 구하시오. 훈련 파일과 겹친 시험 리뷰의 정확도도 따로 구하시오.

① 자르는 방법을 받아 모델을 맞추는 함수 `fit_score( )`를 만듭니다. ② 두 방법으로 맞추어 견줍니다. ③ 겹친 리뷰를 떼어 정확도를 잽니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score

def fit_score(col, **kw):                                    # 자르는 방법을 받아 TF-IDF + 로지스틱 회귀
    v = TfidfVectorizer(lowercase=False, sublinear_tf=True, **kw)
    m = LogisticRegression(C=3, max_iter=3000).fit(v.fit_transform(train[col]), y_tr)
    return v, m, m.predict(v.transform(test[col]))

v_w, lr_w, pred_w = fit_score("document", token_pattern=r"\S+")
print("띄어쓰기 단어  정확도:", round((pred_w == y_te).mean(), 3))
vec, lr, pred = fit_score("m", token_pattern=r"\S+", ngram_range=(1, 2), min_df=2)
print("형태소 1~2개씩 정확도:", round((pred == y_te).mean(), 3), " 조각 수:", len(vec.vocabulary_),
      " 리뷰당 평균:", round(vec.transform(train["m"]).getnnz(axis=1).mean(), 1))
print(confusion_matrix(y_te, pred))
print("정밀도:", round(precision_score(y_te, pred), 3), " 재현율:", round(recall_score(y_te, pred), 3))
ov = test["document"].isin(train0["document"]).values        # 훈련 파일에도 똑같은 글이 있는 시험 리뷰
print("겹친 리뷰", ov.sum(), "개 정확도", round((pred[ov] == y_te[ov]).mean(), 3),
      " 나머지", round((pred[~ov] == y_te[~ov]).mean(), 3))

acc = pd.Series({"찍기": max(y_te.mean(), 1 - y_te.mean()), "띄어쓰기": (pred_w == y_te).mean(),
                 "형태소 1~2개": (pred == y_te).mean()})
acc.plot.bar(color=["#DDDDDD", "#BBBBBB", "#C8234B"], figsize=(4.5, 3))
plt.ylim(0.45, 0.9)
plt.ylabel("시험 정확도")
plt.xticks(rotation=0)
plt.show()

형태소 1~2개씩 자른 모델이 87.1%로, 띄어쓰기(79.3%)보다 7.8%p 높습니다. 조각은 172,906종류인데 리뷰 하나에 든 조각은 평균 33.6개뿐입니다. 17만 칸 가운데 33칸 남짓만 0이 아닌 것입니다. 이런 벡터를 희소 벡터라고 하며, 사이킷런은 0이 아닌 칸만 저장해 메모리를 아낍니다.

> **[코드 읽기] `kw`, `sublinear_tf=True`, `C=3`**
>
> `def fit_score(col, **kw)`의 `**kw`는 "나머지 이름 붙은 값을 모두 받아 두라"는 뜻입니다. 받은 값은 `TfidfVectorizer(..., **kw)`로 그대로 넘어갑니다. `ngram_range=(1, 2)`는 1개와 2개 묶음을 모두 세라는 뜻이고, `min_df=2`는 리뷰 두 개 이상에 나온 조각만 쓰라는 뜻입니다. `sublinear_tf=True`는 4.3의 1 + ln tf입니다. `C=3`은 가중치를 얼마나 자유롭게 키우게 둘지 정하는 값입니다. 작으면 가중치를 0 쪽으로 눌러 단순한 모델이 되고, 크면 훈련 리뷰에 더 맞춥니다. 기본값은 1입니다. 3은 시험 리뷰를 보고 고른 값이 아니라, 훈련 리뷰만으로 교차검증해 고른 값입니다(바로 아래).

**☆ C는 어떻게 골랐나.** 시험 리뷰의 점수를 보며 C를 고르면, 그 점수는 더 이상 처음 보는 리뷰의 성적이 아닙니다. 그래서 10장 [파이썬 실습 10-5]처럼 훈련 리뷰만 다섯 조각으로 나누어 교차검증했습니다. 넷으로 배우고 남은 하나로 재기를 다섯 번 되풀이해 평균을 냅니다. `make_pipeline`으로 TF-IDF와 모델을 묶어, 조각마다 TF-IDF 사전도 새로 배우게 했습니다. 실행에 몇 분이 걸리므로 선택으로 둡니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv5 = StratifiedKFold(5, shuffle=True, random_state=0)       # 훈련 리뷰만 다섯 조각으로 (시험 리뷰는 쓰지 않는다)
for c in [0.3, 1, 3, 10]:
    pipe = make_pipeline(TfidfVectorizer(lowercase=False, sublinear_tf=True, token_pattern=r"\S+",
                                         ngram_range=(1, 2), min_df=2),
                         LogisticRegression(C=c, max_iter=3000))
    s = cross_val_score(pipe, train["m"], y_tr, cv=cv5, n_jobs=-1)   # 조각마다 TF-IDF 사전부터 새로 배운다
    print(f"C={c:<4} 검증 정확도 {s.mean():.4f}")

C = 3의 검증 정확도가 0.8692로 가장 높아 이 값을 썼습니다. 10으로 키우면 0.8675로 거의 같고, 0.3으로 줄이면 0.8541로 떨어집니다. 시험 리뷰는 C를 정한 뒤에 한 번만 썼습니다.

#### [파이썬 실습 11-6 ☆] 자르는 방법과 모델 바꾸어 보기

형태소 1개씩, 글자 1~3개씩으로 잘라 정확도를 견주고, 같은 조각으로 나이브 베이즈를 맞추어 보시오.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

ways = {"형태소 1개씩": ("m", dict(token_pattern=r"\S+")),
        "글자 1~3개씩": ("document", dict(analyzer="char", ngram_range=(1, 3), min_df=2))}
res = {"띄어쓰기 단어": (pred_w == y_te).mean()}
for name, (col, kw) in ways.items():
    v, m, p = fit_score(col, **kw)
    res[name] = (p == y_te).mean()
    print(f"{name}  조각 수 {len(v.vocabulary_):7d}  정확도 {res[name]:.3f}")
res["형태소 1~2개씩"] = (pred == y_te).mean()

cvm = CountVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, lowercase=False)
nb = MultinomialNB().fit(cvm.fit_transform(train["m"]), y_tr)
acc_nb = (nb.predict(cvm.transform(test["m"])) == y_te).mean()
print("나이브 베이즈(형태소 1~2개, 개수) 정확도:", round(acc_nb, 3))

pd.Series(res).plot.barh(color="#BBBBBB", figsize=(6, 3))
plt.axvline(acc_nb, color="#C8234B", ls="--", label="나이브 베이즈")
plt.xlim(0.75, 0.9)
plt.xlabel("시험 정확도 (로지스틱 회귀)")
plt.legend()
plt.show()

글자 1~3개씩 자른 모델(87.2%)은 형태소 분석기 없이도 형태소 1~2개씩(87.1%)과 거의 같았습니다. 나이브 베이즈는 85.8%였습니다. `analyzer="char"`는 띄어쓰기를 무시하고 글자 단위로 자르라는 뜻입니다.

#### [파이썬 실습 11-7 ★] 모델이 배운 단어와 새 문장

가중치가 가장 큰 조각과 가장 작은 조각을 8개씩 찾고, 처음 보는 문장 여섯 개의 긍정 확률을 구하시오.

① `coef_`에서 가중치를 꺼내 크기 순서로 정렬합니다. ② 새 문장을 형태소로 잘라 `predict_proba( )`에 넣습니다. ③ 양 끝 6개씩을 막대그래프로 그립니다.

In [ ]:
names = vec.get_feature_names_out()
coef = lr.coef_[0]
order = np.argsort(coef)                                     # 가중치가 작은 것부터의 순서
print("긍정 쪽:", [(names[i], round(float(coef[i]), 1)) for i in order[::-1][:8]])
print("부정 쪽:", [(names[i], round(float(coef[i]), 1)) for i in order[:8]])

new = ["배우들 연기가 정말 최고였어요", "시간이 아까운 영화", "재미없을 줄 알았는데 의외로 재밌네요",
       "기대 안 했는데 대박", "재미없지는 않았다", "그냥 그랬어요"]
p_new = lr.predict_proba(vec.transform(morphs(new)))[:, 1]
for s, p in zip(new, p_new):
    print(f"긍정 확률 {p:.3f}  {s}")

pick = np.r_[order[:6], order[-6:]]
plt.figure(figsize=(6, 4))
plt.barh(range(12), coef[pick], color=["#BBBBBB"] * 6 + ["#C8234B"] * 6)
plt.yticks(range(12), names[pick])
plt.xlabel("가중치 (왼쪽 = 부정, 오른쪽 = 긍정)")
plt.show()

"최고"(+13.0)와 "최악"(−14.1)이 양 끝에 있습니다. 새 문장 가운데 "재미없지는 않았다"(0.034)와 "그냥 그랬어요"(0.071)가 부정 쪽으로 기울었습니다. 3.6에서 본 그대로입니다.

> **[코드 읽기] `np.argsort( )`와 `[::-1]`**
>
> `np.argsort(coef)`는 값을 작은 것부터 늘어놓았을 때 "몇 번째 칸이 오는지"를 돌려줍니다. `order[:8]`은 가장 작은 8개, `order[::-1][:8]`은 순서를 뒤집은 뒤 앞 8개, 곧 가장 큰 8개입니다.

#### [파이썬 실습 11-8 ★] 확신과 망설임, 그리고 틀린 리뷰

시험 리뷰마다 긍정 확률을 구해 확신한 리뷰(0.9 이상, 0.1 이하)와 망설인 리뷰(0.4~0.6)의 비율과 정확도를 구하시오. 12자 이하의 짧은 리뷰 가운데 모델이 가장 확신하고 틀린 것도 찾아보시오.

① 긍정 확률을 구합니다. ② 확신과 망설임을 True/False로 나누어 셉니다. ③ 틀린 짧은 리뷰를 확률 순서로 뽑습니다.

In [ ]:
prob = lr.predict_proba(vec.transform(test["m"]))[:, 1]       # 시험 리뷰마다 긍정 확률
sure = (prob >= 0.9) | (prob <= 0.1)                          # 확신하는 답
unsure = (prob > 0.4) & (prob < 0.6)                          # 망설이는 답
print(f"확신: 리뷰의 {sure.mean():.3f}, 정확도 {(pred[sure] == y_te[sure]).mean():.3f}")
print(f"망설임: 리뷰의 {unsure.mean():.3f}, 정확도 {(pred[unsure] == y_te[unsure]).mean():.3f}")
print("틀린 리뷰:", (pred != y_te).sum(), "개")

wrong = test.assign(prob=prob)[pred != y_te].drop_duplicates("document")
short = wrong[wrong["document"].str.len() <= 12]
print(short[short["label"] == 0].nlargest(4, "prob")[["document", "prob"]].round(3).to_string(index=False))
print(short[short["label"] == 1].nsmallest(4, "prob")[["document", "prob"]].round(3).to_string(index=False))

plt.figure(figsize=(6, 3))
plt.hist([prob[y_te == 0], prob[y_te == 1]], bins=20, stacked=True,
         color=["#BBBBBB", "#C8234B"], label=["실제 부정", "실제 긍정"])
plt.xlabel("모델이 낸 긍정 확률")
plt.ylabel("시험 리뷰 수")
plt.legend()
plt.show()

확신한 53.1%의 정확도는 97.1%, 망설인 8.7%는 55.4%입니다. 확신하고 틀린 짧은 리뷰 위쪽 표는 부정 라벨인데 긍정 확률이 높은 것, 아래쪽 표는 긍정 라벨인데 확률이 낮은 것입니다. "좋다"가 부정이고 "별루"가 긍정인 것은 모델이 아니라 라벨의 문제로 보입니다.

#### [파이썬 실습 11-9 ☆] 훈련 리뷰 수에 따른 정확도

훈련 리뷰를 무작위로 1,000개, 5,000개, 2만 개, 5만 개만 골라 다시 배우고 시험 정확도를 견주시오.

In [ ]:
rng = np.random.RandomState(0)
perm = rng.permutation(len(train))                           # 훈련 리뷰를 섞은 순서
curve = {}
for n in [1000, 5000, 20000, 50000]:
    idx = perm[:n]
    v = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2 if n > 5000 else 1,
                        lowercase=False, sublinear_tf=True)
    m = LogisticRegression(C=3, max_iter=3000).fit(v.fit_transform(train["m"].values[idx]), y_tr[idx])
    curve[n] = (m.predict(v.transform(test["m"])) == y_te).mean()
curve[len(train)] = (pred == y_te).mean()
for n, a in curve.items():
    print(f"훈련 리뷰 {n:6d}개  정확도 {a:.3f}")

pd.Series(curve).plot(marker="o", logx=True, color="#C8234B", figsize=(5, 3))
plt.xlabel("훈련 리뷰 수 (눈금은 10배씩)")
plt.ylabel("시험 정확도")
plt.show()

1,000개에서 5,000개로 늘릴 때는 5.2%p가 올랐습니다. 5만 개에서 146,025개로 늘릴 때는 1.5%p만 올랐습니다. 가로축을 10배씩 눈금으로 그리면 거의 직선이 됩니다. 정확도를 같은 만큼 올리려면 데이터가 몇 배씩 필요하다는 뜻입니다.

#### [파이썬 실습 11-10 ☆] 같은 뜻 다른 단어: TF-IDF와 Word2Vec

TF-IDF 코사인 유사도와 Word2Vec 유사도를 견주고, 리뷰마다 단어 좌표를 평균해 로지스틱 회귀를 맞추시오. "배우들 연기가 정말 좋았다"와 가장 비슷한 리뷰도 찾으시오.

In [ ]:
from gensim.models import Word2Vec

v1 = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True).fit(train["m"])
for a, b in [("연기 최고", "연기력 훌륭"), ("연기 최고", "연기 최악")]:
    s = cosine_similarity(v1.transform(morphs([a])), v1.transform(morphs([b])))[0, 0]
    print(f"TF-IDF 코사인  {a} / {b}: {s:.3f}")

sents = train["m"].str.split().tolist()
w2v = Word2Vec(sents, vector_size=100, window=5, min_count=5, sg=1, epochs=5, seed=42, workers=1)
for w in ["배우", "지루", "최고"]:
    print(w, "->", [(x, round(float(s), 2)) for x, s in w2v.wv.most_similar(w, topn=4)])
for a, b in [("지루", "따분"), ("연기", "연기력"), ("최고", "최악")]:
    print(f"Word2Vec 유사도  {a} / {b}: {w2v.wv.similarity(a, b):.3f}")

def doc_vec(m):                                              # 리뷰 = 단어 벡터의 평균
    vs = [w2v.wv[w] for w in m.split() if w in w2v.wv]
    return np.mean(vs, axis=0) if vs else np.zeros(100)
Xe = np.array([doc_vec(m) for m in train["m"]])
Te = np.array([doc_vec(m) for m in test["m"]])
lre = LogisticRegression(max_iter=3000).fit(Xe, y_tr)
print("평균 벡터 + 로지스틱 회귀 정확도:", round((lre.predict(Te) == y_te).mean(), 3))
print("왕 - 남자 + 여자 ->", [x for x, _ in w2v.wv.most_similar(positive=["왕", "여자"], negative=["남자"], topn=3)])

q = "배우들 연기가 정말 좋았다"                              # 가장 비슷한 리뷰 찾기
sim = cosine_similarity(v1.transform(morphs([q])), v1.transform(train["m"]))[0]
for i in np.argsort(-sim)[:3]:
    print(f"{sim[i]:.3f}  {train['document'].values[i]}")

Word2Vec은 "최고"의 이웃으로 "쵝오", "단연", "최강"을 찾았습니다. "왕 − 남자 + 여자"는 "여왕"이 아니라 엉뚱한 단어들이었습니다(6.1). 컴퓨터마다 결과가 조금 다를 수 있어 `seed`를 고정하고 `workers=1`로 두었습니다.

#### [파이썬 실습 11-11 ☆] 연방주의자 논문: 누가 썼을까

연방주의자 논문 85편을 읽어 저자별로 "upon", "whilst", "on"이 1,000단어마다 몇 번 나오는지 구하시오. 기능어 31개로 나이브 베이즈를 맞추어 논란의 12편도 판정하시오.

In [ ]:
import re
import urllib.request
import warnings
import pyreadr
warnings.filterwarnings("ignore", category=RuntimeWarning)   # 날짜 칸을 읽을 때 나는 경고 숨기기

urllib.request.urlretrieve("https://raw.githubusercontent.com/cran/corpus/master/data/federalist.rda",
                           "federalist.rda")
fed = pyreadr.read_r("federalist.rda")["federalist"]
fed["no"] = fed["name"].str.extract(r"(\d+)").astype(int)
DISP = [49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 63]      # 저자 논란이 있는 12편
fed["저자"] = np.where(fed["no"].isin(DISP), "논란", fed["author"].fillna("공저"))
print(fed["저자"].value_counts().to_dict())

def per1000(texts, word):                                    # 1,000단어당 횟수
    words = re.findall(r"[a-z]+", " ".join(texts).lower())
    return 1000 * words.count(word) / len(words)
fw = pd.DataFrame({g: [per1000(fed.loc[fed["저자"] == g, "text"], w) for w in ["upon", "whilst", "on"]]
                   for g in ["Hamilton", "Madison", "논란"]}, index=["upon", "whilst", "on"])
print(fw.round(2).to_string())
print("upon이 없는 논란의 글:", sum(per1000([t], "upon") == 0 for t in fed.loc[fed["저자"] == "논란", "text"]), "편")

FW = ["upon", "whilst", "while", "on", "by", "there", "although", "also", "enough", "an", "of",
      "to", "would", "may", "this", "the", "and", "in", "a", "is", "it", "that", "which", "be",
      "as", "with", "for", "from", "at", "or", "not"]           # 내용과 상관없는 기능어 31개
cvf = CountVectorizer(vocabulary=FW, token_pattern=r"[a-z]+")
known = fed[fed["저자"].isin(["Hamilton", "Madison"])]
nbf = MultinomialNB().fit(cvf.transform(known["text"]), known["저자"])
dd = fed[fed["저자"] == "논란"].sort_values("no")
print(dict(zip(dd["no"].tolist(), nbf.predict(cvf.transform(dd["text"])).tolist())))

fw.plot.bar(color=["#BBBBBB", "#C8234B", "#555555"], figsize=(5.5, 3))
plt.ylabel("1,000단어당 횟수")
plt.xticks(rotation=0)
plt.show()

lam = fw.loc["upon", ["Hamilton", "Madison"]] * 2           # 2,000단어 글에서 기대하는 "upon" 횟수
p0 = np.exp(-lam)                                            # 한 번도 안 나올 확률 e^(-λ)
print("기대 횟수", lam.round(2).to_dict(), " 0번일 확률", p0.round(5).to_dict())
print("매디슨 쪽으로", round(p0["Madison"] / p0["Hamilton"]), "배")

논란의 12편 가운데 10편에는 "upon"이 한 번도 없었습니다. 나이브 베이즈는 11편을 매디슨, 55번을 해밀턴으로 판정했습니다. 마지막 두 줄은 4.4의 [더 알아보기]에서 쓴 "0번일 확률"입니다. 모스텔러와 월리스는 55번도 매디슨 쪽으로 보았습니다. 단어 31개를 독립이라고 보고 곱한 단순한 모델의 한계로 보입니다.

> **핵심 3줄**
>
> 텍스트 실습은 읽기 → 형태소로 자르기 → TF-IDF → 로지스틱 회귀 → 확률과 가중치 읽기 순서로 흐릅니다.
>
> ★ 실습만 돌려도 03절의 79.3%, 87.1%, 0.034, 53.1%와 97.1%가 모두 나옵니다.
>
> 글자 1~3개씩(87.2%)은 형태소 분석기 없이도 거의 같은 성적을 냈고, 훈련 리뷰를 늘리는 효과는 점점 줄었습니다.

## [미니 프로젝트 11] 영화 리뷰로 배운 모델이 쇼핑 리뷰의 마음도 읽을까


**질문.** 동네 온라인 가게 주인이 리뷰를 긍정과 부정으로 나누고 싶어 합니다. 이 장에서 만든 87.1%짜리 영화 리뷰 모델을 그대로 쓰면 될까요? 데이터 과학자 스티븐 스키에나는 감성 분석 회사를 함께 세웠다가, 고객마다 업종이 달라 원하는 답이 달랐던 경험을 적었습니다. 모델보다 "누구의 어떤 질문인가"가 먼저라는 이야기입니다. 이 프로젝트에서는 그 질문을 숫자로 확인합니다.

**데이터.** 네이버 쇼핑 리뷰 20만 개입니다. 키위를 만든 이민철이 2020년 6~7월에 네이버 쇼핑의 상품 후기를 별점과 함께 모았습니다. 한 행은 리뷰 하나이고, 열은 별점(1, 2, 4, 5)과 리뷰 글, 두 개입니다. 3점은 빠졌고, 긍정(4~5점)과 부정(1~2점)이 반반이 되도록 뽑았습니다. 영화 리뷰와 같은 방식이니 라벨을 같은 뜻으로 볼 수 있습니다.

**1단계: 데이터 신원 확인**([파이썬 실습 11-1] 응용). 데이터를 읽고 별점별 개수를 센 뒤, 4~5점을 긍정(1)으로 하는 라벨을 만드시오. 라벨이 엇갈린 글과 겹친 글을 정리하시오.

In [ ]:
URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
shop = pd.read_csv(URL, sep="\t", header=None, names=["별점", "리뷰"], quoting=3)
print(shop.shape)
print(shop["별점"].value_counts().sort_index().to_dict())
shop["label"] = (shop["별점"] >= 4).astype(int)              # 4~5점 긍정, 1~2점 부정
n_lab = shop.groupby("리뷰")["label"].nunique()
shop = shop[~shop["리뷰"].isin(n_lab[n_lab > 1].index)].drop_duplicates("리뷰")
print("정리 후:", len(shop), " 긍정 비율:", round(shop["label"].mean(), 3))
print("길이 가운데 값:", int(shop["리뷰"].str.len().median()), "자")

shop["별점"].value_counts().sort_index().plot.bar(color="#BBBBBB", figsize=(4, 3))
plt.xlabel("별점")
plt.ylabel("리뷰 수")
plt.xticks(rotation=0)
plt.show()

정리하고 나니 199,898개가 남았고, 긍정 비율은 0.5입니다. 리뷰 길이의 가운데 값은 29자로 영화 리뷰(27자)와 비슷합니다.

**2단계: 기준선과 영화 모델**([파이썬 실습 11-3], [11-5] 응용). 75%로 나누어 25%를 시험 리뷰로 두고, 영화 리뷰로 배운 모델을 그대로 적용하시오.

In [ ]:
from sklearn.model_selection import train_test_split

s_tr, s_te = train_test_split(shop, test_size=0.25, random_state=42, stratify=shop["label"])
s_tr, s_te = s_tr.copy(), s_te.copy()
s_tr["m"] = morphs(s_tr["리뷰"].tolist())
s_te["m"] = morphs(s_te["리뷰"].tolist())
ys = s_te["label"].values
print("시험 리뷰:", len(s_te), " 기준선:", round(max(ys.mean(), 1 - ys.mean()), 3))
p_movie = lr.predict(vec.transform(s_te["m"]))               # 영화 리뷰로 배운 모델 그대로
print("영화 모델의 정확도:", round((p_movie == ys).mean(), 3))
print(confusion_matrix(ys, p_movie))

영화 모델은 쇼핑 리뷰에서 79.5%를 맞혔습니다. 찍기(50.0%)보다는 낫지만, 영화 리뷰에서의 87.1%보다 7.6%p 낮습니다. 혼동행렬을 보면 부정 리뷰 7,715개를 긍정으로 읽었습니다. 틀림이 한쪽으로 몰렸습니다.

**3단계: 쇼핑 리뷰로 배우기**([파이썬 실습 11-5] 응용). 쇼핑 리뷰로 같은 모델을 배우고, 쇼핑 리뷰 1,000개만으로 배운 모델과도 견주시오.

In [ ]:
vs = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, lowercase=False, sublinear_tf=True)
lrs = LogisticRegression(C=3, max_iter=3000).fit(vs.fit_transform(s_tr["m"]), s_tr["label"])
p_shop = lrs.predict(vs.transform(s_te["m"]))
print("쇼핑 모델의 정확도:", round((p_shop == ys).mean(), 3))

few = s_tr.sample(1000, random_state=0)                      # 쇼핑 리뷰 1,000개만
v_few = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), lowercase=False, sublinear_tf=True)
m_few = LogisticRegression(C=3, max_iter=3000).fit(v_few.fit_transform(few["m"]), few["label"])
acc_few = (m_few.predict(v_few.transform(s_te["m"])) == ys).mean()
print("쇼핑 리뷰 1,000개로 배운 모델:", round(acc_few, 3))

board = pd.Series({"찍기": 0.5, "영화 리뷰\n14만 개": (p_movie == ys).mean(),
                   "쇼핑 리뷰\n1,000개": acc_few, "쇼핑 리뷰\n15만 개": (p_shop == ys).mean()})
board.plot.bar(color=["#DDDDDD", "#BBBBBB", "#C8234B", "#C8234B"], figsize=(5.5, 3))
plt.ylim(0.45, 1)
plt.ylabel("쇼핑 리뷰 시험 정확도")
plt.xticks(rotation=0)
plt.show()

쇼핑 리뷰로 배운 모델은 92.3%입니다. 놀라운 것은 쇼핑 리뷰 1,000개만으로 배운 모델(87.1%)이 영화 리뷰 14만 개로 배운 모델보다 7.6%p 높다는 점입니다. 3.9에서 리뷰를 늘리는 효과가 점점 줄었던 것과 견주어 보십시오. 많은 데이터보다 같은 세상의 데이터가 중요했습니다.

**4단계: 어디서 틀렸나 보기**([파이썬 실습 11-7] 응용). "배송", "사이즈", "반품", "재구매"가 든 리뷰의 정확도를 두 모델로 견주고, 짧은 문장 네 개의 긍정 확률을 구하시오.

In [ ]:
for w in ["배송", "사이즈", "반품", "재구매"]:
    has = s_te["리뷰"].str.contains(w).values
    print(f"{w:4s} {has.sum():5d}개  영화 모델 {(p_movie[has] == ys[has]).mean():.3f}"
          f"  쇼핑 모델 {(p_shop[has] == ys[has]).mean():.3f}")
probe = ["배송이 빨라요", "생각보다 작아요", "반품했어요", "재구매 의사 있어요"]
pm = lr.predict_proba(vec.transform(morphs(probe)))[:, 1]
ps = lrs.predict_proba(vs.transform(morphs(probe)))[:, 1]
for s, a, b in zip(probe, pm, ps):
    print(f"영화 모델 {a:.3f}  쇼핑 모델 {b:.3f}  {s}")

영화 모델은 "반품했어요"에 긍정 확률 0.874를 주었습니다. 영화 리뷰에는 반품이 없으니 이 말이 나쁜 소식이라는 것을 배울 수 없었습니다. "생각보다 작아요"도 0.695였습니다. 쇼핑 모델은 두 문장을 0.005와 0.066으로 읽었습니다. "사이즈"가 든 리뷰에서 두 모델의 차이(76.7%와 92.1%)가 가장 컸습니다.

**5단계: 한계 쓰기.** 이 모델이 아직 모르는 것을 [표 11-2]처럼 세 가지 이상 쓰시오. 예를 들어 다른 쇼핑몰, 3점 리뷰, 2020년 이후의 말에 대해 생각해 보십시오.

**제출물.** 그림 2장(별점별 리뷰 수, 세 모델의 정확도 비교), 숫자 표 1개(단어별 두 모델의 정확도), 결론 5문장 이내.

**[표 11-3] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 출처, 라벨 규칙(3점 제외, 반반 추출), 엇갈린 라벨을 모두 밝힘 | 빈칸과 중복만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 찍기, 영화 모델, 쇼핑 모델을 같은 시험 리뷰로 견줌 | 쇼핑 모델과 찍기만 견줌 | 기준선 없이 정확도만 씀 |
| 기법 적용(30%) | 형태소 자르기, TF-IDF, 로지스틱 회귀, 1,000개 모델이 모두 맞음 | 모델은 맞으나 영화 모델 적용이나 1,000개 비교가 빠짐 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 틀린 문장을 예로 들어 왜 틀렸는지 설명하고 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 정리하면 20만 개 가운데 199,898개가 남고 긍정 비율은 0.5입니다. 시험 리뷰 49,975개에서 찍기는 50.0%, 영화 모델은 79.5%, 쇼핑 모델은 92.3%, 쇼핑 리뷰 1,000개 모델은 87.1%입니다. 영화 모델의 틀림은 부정을 긍정으로 읽은 쪽(7,715개)이 그 반대(2,541개)의 세 배쯤입니다. 단어별로는 "사이즈"(76.7% 대 92.1%)와 "반품"(85.3% 대 99.3%)에서 차이가 큽니다. 한계로는 다음 셋을 기대합니다. 첫째, 2020년 네이버 쇼핑 한 곳의 글이라 다른 쇼핑몰이나 요즘 말에는 맞지 않을 수 있습니다. 둘째, 3점 리뷰를 뺐으니 실제 가게에서 가장 애매한 글은 시험한 적이 없습니다. 셋째, 긍정과 부정을 반반으로 뽑았지만 실제 가게의 리뷰는 긍정이 훨씬 많아 같은 정확도라도 뜻이 달라집니다.

## ✏️ [연습문제 11-6] 망설이는 리뷰를 사람에게

[파이썬 실습 11-8]에 이어, 모델의 긍정 확률이 0.3보다 크고 0.7보다 작은 리뷰를 사람에게 넘긴다고 합시다. 넘기는 리뷰의 수와 비율, 모델이 맡은 리뷰의 정확도, 넘긴 리뷰를 모델이 판정했을 때의 정확도를 구하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
hand = (prob > 0.3) & (prob < 0.7)                           # 사람에게 넘길 리뷰
print("사람에게 넘기는 리뷰:", hand.sum(), "개, 비율", round(hand.mean(), 3))
print("모델이 맡은 리뷰의 정확도:", round((pred[~hand] == y_te[~hand]).mean(), 3))
print("넘긴 리뷰를 모델이 판정했다면:", round((pred[hand] == y_te[hand]).mean(), 3))

## 부록 · 04절의 식을 코드로 확인하기

책 04절에 나온 짧은 확인 코드입니다. 손 계산의 답과 같은지 보십시오.

**4.1 idf와 로그** (`np.log`는 자연로그 ln)

In [ ]:
N, df = 3, np.array([2, 1])                  # "배우"는 리뷰 2개에, "최고"는 1개에 나온다
print("idf:", np.log(N / df).round(3))       # np.log는 자연로그 ln
print("ln 10, ln 100:", np.log([10, 100]).round(3))
print("사이킷런 idf:", (np.log((1 + N) / (1 + df)) + 1).round(3))

**4.2 [예제 11-1]의 내적, 길이, 코사인 유사도** (`@`는 내적, `np.linalg.norm`은 길이)

In [ ]:
a, b = np.log(3 / 2), np.log(3)               # idf: 리뷰 2개에 나온 단어, 1개에 나온 단어
d1 = np.array([a, 0, a, 0, b])               # (배우, 스토리, 연기, 지루, 최고)
d2 = np.array([a, 0, a, a, 0])
dot = d1 @ d2                                # 내적: 같은 칸끼리 곱해 모두 더하기
n1, n2 = np.linalg.norm(d1), np.linalg.norm(d2)
print("내적", round(dot, 3), " 길이", round(n1, 3), round(n2, 3), " 코사인", round(dot / (n1 * n2), 3))

**[잠깐만] 코사인 유사도와 상관계수**: (0, 1, 2, 3)과 모두 100씩 큰 벡터

In [ ]:
u = np.array([0, 1, 2, 3])
w = u + 100                                  # 모두 100씩 크다
cos = u @ w / (np.linalg.norm(u) * np.linalg.norm(w))
print("코사인:", round(cos, 3), " 상관계수:", round(np.corrcoef(u, w)[0, 1], 3))

**4.4 "재미없지는 않았다"를 조각별 몫으로 나누기** (실습 11-7을 돌린 뒤 실행)

In [ ]:
x = vec.transform(morphs(["재미없지는 않았다"]))            # 이 리뷰의 TF-IDF 무게(0이 아닌 칸만)
part = pd.Series(coef[x.indices] * x.data, index=names[x.indices]).sort_values()   # 가중치 × 무게
print(part.round(3).to_dict())
print("절편 + 합 =", round(lr.intercept_[0] + part.sum(), 3))